# Week 3 — V1 Data Pipeline for MobiliFAI
**Student:** Sang Yen Ling (A24AI0126)

**Course:** AI Engineering (SAIA 3113)

**Pathway:** A (PDF/TXT) only — MobiliFAI is a text/document RAG project. Web pages are ingested as dated `.txt` snapshots.

**Flow:** Source → Ingest → Inspect → Prepare (clean + page-aware chunk + requirement tagging) → Validate → Record lineage.
Run order: Cell 1 → upload to `data/raw` → Cell 3 → Cell 4 → Cell 4A → A1/A3 → B-compare → B-export → B3 → B4.1 → 6A → 6B → 7 → 8A → 9.

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW = RAW_DIR; PROCESSED = PROCESSED_DIR; BASE = BASE_DIR; OUT = PROCESSED_DIR
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


**Upload layout expected under `data/raw/`:** `pdf/` (4 PDFs) and `web/` (3 `.txt` snapshots). Raw files are *not* committed to GitHub (copyright / rights not verified).

In [2]:
# Cell 3 — Verify uploaded inputs
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 7
pdf/FAI_Academic_Handbook_2025_2026.pdf bytes: 20516657
pdf/UTM_Academic_Calendar_20262027.pdf bytes: 627618
pdf/UTM_Academic_Rules_and_Regulations.pdf bytes: 867018
pdf/UTM_International_Outbound_Pre-Departure_Briefing.pdf bytes: 1103512
web/jasso_student_exchange_support.txt bytes: 2271
web/shisu_china_link_2026.txt bytes: 3137
web/utm_mobility_portal.txt bytes: 5619
Upload verified. Select your relevant input(s) for Step A3.


In [3]:
# Cell 4 — Source selection + lineage utilities
from pathlib import Path
import json, hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(p for p in RAW_DIR.rglob("*") if p.is_file()
                     and (filename is None or p.name == filename)
                     and (suffixes is None or p.suffix.lower() in suffixes))
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {"pipeline_version": "week3-v1",
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "records": records}
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")

Cell 4 completed successfully.


In [4]:
# Cell 4A — Common lineage registry
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve(); output = Path(output).resolve()
    if not source.is_file(): raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file(): raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0: raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    source.relative_to(RAW_DIR.resolve()); output.relative_to(PROCESSED_DIR.resolve())
    lineage_entries[(str(source), str(output))] = {"source": source, "output": output,
                                                   "processing": processing.strip()}
    print("Registered:", source.name, "->", output.name)
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


## Part A — Start with the data
**A1 requirement → source map** (from `week2_requirements.md`)

| Requirement | Needed information | Source(s) | Constraint |
|---|---|---|---|
| FR1 | Procedures, eligibility, deadlines, pre-departure steps | S02 Briefing, S05 Portal, S06 JASSO, S07 SISU | Portal listings are 2022 (stale); JASSO "as of Jan 2024" |
| FR1 / FR4 | Credit, registration & academic rules; term dates / holidays (availability) | S01 Handbook, S03 Calendar 2026/27, S04 Rules | Rules are Bahasa Malaysia; Handbook calendar pages are images |
| FR3 / RAI1 | Title, URL, **page number** for citations | All (metadata in catalogue) | Page must survive chunking |
| RAI3 / FR4 | Version / validity / staleness to flag outdated sources | All (`valid_for`, `max_year`) | Years are heuristic |
| NFR4 | Re-runnable update of one source | Pipeline + SHA-256 manifest | — |

In [5]:
# Cell A1 — Source catalogue (identity, owner, validity, rights)
REFERENCE_YEAR = 2026   # fixed so reruns are deterministic
RIGHTS = "Not verified; all-rights-reserved notice -> keep repo private, do not redistribute"
SOURCES = [
 dict(id="S01", declared_year=2025, rel="pdf/FAI_Academic_Handbook_2025_2026.pdf", kind="handbook",
      title="FAI Academic Handbook 2025-2026", owner="Faculty of Artificial Intelligence, UTM",
      url="https://drive.google.com/file/d/1coXIUom2QVI_YlUtQ0CjFeNkZU2xOPy3/view",
      valid_for="2025/2026 session", language="en", access=RIGHTS,
      requirement="FR1, FR3", note="printed page = PDF page - 5; PDF pp.53-56 (calendar 2025/26) are images"),
 dict(id="S02", declared_year=2023, rel="pdf/UTM_International_Outbound_Pre-Departure_Briefing.pdf", kind="briefing",
      title="UTM Outbound Mobility Pre-Departure Briefing", owner="UTM International",
      url="https://international.utm.my/wp-content/uploads/2023/05/UTM-OUTBOUND-PREDEPARTURE-BRIEFING.pdf",
      valid_for="undated (uploaded 2023/05)", language="en", access=RIGHTS,
      requirement="FR1, FR3", note="slide deck; page = slide number"),
 dict(id="S03", declared_year=2026, rel="pdf/UTM_Academic_Calendar_20262027.pdf", kind="calendar",
      title="UTM Academic Calendar 2026/2027", owner="UTM (Academic Management Division)",
      url="not recorded (user-supplied PDF)", valid_for="2026/2027 session", language="en+ms", access=RIGHTS,
      requirement="FR1 (availability / holidays), RAI3", note="2 pages: calendar table + weekly Malay table"),
 dict(id="S04", declared_year=2022, rel="pdf/UTM_Academic_Rules_and_Regulations.pdf", kind="rules",
      title="Peraturan Akademik Program Sarjana Muda (Edisi Keenam Belas, 2022)", owner="UTM DVC (Academic & International)",
      url="not recorded (user-supplied PDF)", valid_for="16th edition, 2022", language="ms", access=RIGHTS,
      requirement="FR1, FR4 (credit transfer rules)", note="Bahasa Malaysia; may be superseded by newer edition"),
 dict(id="S05", declared_year=2022, rel="web/utm_mobility_portal.txt", kind="web",
      title="UTM Mobility Portal", owner="UTM International",
      url="https://international.utm.my/utm-mobility-portal-2/", valid_for="snapshot 2026-11-16; listings are 2022",
      language="en", access="Public webpage; snapshot for coursework", requirement="FR1, RAI3", retrieved="2026-11-16",
      note="web snapshot, main content only"),
 dict(id="S06", declared_year=2024, rel="web/jasso_student_exchange_support.txt", kind="web",
      title="JASSO Student Exchange Support Program", owner="JASSO (Japan)",
      url="https://www.jasso.go.jp/en/ryugaku/scholarship_j/ukeire.html", valid_for="page says 'as of Jan 31, 2024'",
      language="en", access="Public webpage; snapshot for coursework", requirement="FR1", note="web snapshot"),
 dict(id="S07", declared_year=2026, rel="web/shisu_china_link_2026.txt", kind="web",
      title="SISU China Link Scholarship 2026 Application Guide", owner="Shanghai International Studies University",
      url="https://www.oisa.shisu.edu.cn/index.php/en/index/newscontent/cid/130/id/663.html",
      valid_for="page dated 2026-01-28", language="en", access="Public webpage; snapshot for coursework",
      requirement="FR1", note="web snapshot"),
]
BY_REL = {s["rel"]: s for s in SOURCES}
print(len(SOURCES), "sources catalogued")

7 sources catalogued


In [6]:
# Cell A3 — Ingest + inspect every source (readable text, pages, empty pages)
import logging, re
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

def load_pages(path):
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        r = PdfReader(str(path))
        return [(i + 1, p.extract_text() or "") for i, p in enumerate(r.pages)]
    if path.suffix.lower() == ".txt":
        txt = path.read_text(encoding="utf-8")
        if txt.startswith("SNAPSHOT of") and "\n\n" in txt:
            txt = txt.split("\n\n", 1)[1]   # drop capture header; retrieval date is kept in the catalogue
        return [(1, txt)]
    raise ValueError("Pathway A needs a PDF or TXT source")

INSPECTION = {}
for s in SOURCES:
    f = select_source(relative_path=s["rel"])
    pages = load_pages(f)
    empty = [p for p, t in pages if len(t.strip()) < 30]
    INSPECTION[s["id"]] = {"pages": len(pages), "chars": sum(len(t) for _, t in pages), "empty_pages": empty}
    print(f'{s["id"]} {f.name[:48]:48} pages={len(pages):3} chars={INSPECTION[s["id"]]["chars"]:7} empty_pages={empty}')

# Representative content (not just "file loaded")
f2 = load_pages(select_source(relative_path="pdf/UTM_International_Outbound_Pre-Departure_Briefing.pdf"))
print("\nS02 slide 4:", repr(f2[3][1][:260]))
f5 = load_pages(select_source(relative_path="web/utm_mobility_portal.txt"))
print("S05 snapshot:", repr(f5[0][1][400:660]))

S01 FAI_Academic_Handbook_2025_2026.pdf              pages= 63 chars= 170996 empty_pages=[53, 54, 55, 56]


S02 UTM_International_Outbound_Pre-Departure_Briefin pages= 11 chars=   5447 empty_pages=[]


S03 UTM_Academic_Calendar_20262027.pdf               pages=  2 chars=   8362 empty_pages=[]


S04 UTM_Academic_Rules_and_Regulations.pdf           pages= 84 chars= 106047 empty_pages=[]
S05 utm_mobility_portal.txt                          pages=  1 chars=   5491 empty_pages=[]
S06 jasso_student_exchange_support.txt               pages=  1 chars=   2116 empty_pages=[]
S07 shisu_china_link_2026.txt                        pages=  1 chars=   2977 empty_pages=[]



S02 slide 4: '4\n• Apply for study abroad\napproval from DVC\n(Academic &\nInternational), UTM.\n• Application MUST be\nsubmitted at least 3\nmonths before the\ndeparture date\n• UTM International will\nassist student with the\napplication approval\nincluding the budget\nrequired (if an'
S05 snapshot: ' student who wishes to undergo any of the outbound mobility programmes must submit the UTM outbound mobility application form. The student is required to print and submit the hardcopy version of the Study Abroad, Research Internship Abroad, International Invit'


## Part B — Prepare the data
Compare three configurations on the same sources, then pick one. Metrics are simple proxies:
*page-traceable* (FR3 needs page for citations) and *starts lowercase* (proxy for a chunk beginning mid-sentence).

In [7]:
# Cell B-compare — fixed vs paragraph-aware (lab) vs page-aware (project)
SIZE = 800

def clean_text(t):
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r" ?\n ?", "\n", t)
    return t.strip()

def fixed_chunks(s, size=SIZE):
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=SIZE):
    parts, current = [], ""
    for seg in re.split(r"\n\s*\n", s):
        seg = seg.strip()
        if not seg: continue
        if len(seg) > size:
            if current: parts.append(current); current = ""
            parts.extend(fixed_chunks(seg, size))
        elif len(current) + len(seg) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + seg
        else:
            if current: parts.append(current)
            current = seg
    if current: parts.append(current)
    return parts

def page_aware_chunks(pages, size=SIZE):
    """Chunk inside each page so every chunk keeps its page number."""
    out = []
    for page, text in pages:
        text = clean_text(text)
        if not text: continue
        units = []
        for seg in re.split(r"\n\s*\n", text):
            seg = seg.strip()
            if not seg: continue
            if len(seg) <= size: units.append(seg); continue
            for line in seg.split("\n"):
                line = line.strip()
                while len(line) > size:
                    units.append(line[:size]); line = line[size:]
                if line: units.append(line)
        cur = ""
        for u in units:
            if cur and len(cur) + len(u) + 1 > size:
                out.append((page, cur)); cur = u
            else:
                cur = (cur + "\n" + u) if cur else u
        if cur: out.append((page, cur))
    return out

stats = {"fixed": [], "paragraph-aware": [], "page-aware": []}
for s in SOURCES:
    pages = load_pages(select_source(relative_path=s["rel"]))
    whole = clean_text("\n".join(t for _, t in pages))
    stats["fixed"] += [(None, c) for c in fixed_chunks(whole)]
    stats["paragraph-aware"] += [(None, c) for c in paragraph_chunks(whole)]
    stats["page-aware"] += page_aware_chunks(pages)

print(f'{"method":16}{"chunks":>8}{"page-traceable":>16}{"starts lowercase":>18}{"avg chars":>11}')
for m, rows in stats.items():
    n = len(rows)
    pt = sum(p is not None for p, _ in rows) / n
    lc = sum(c[:1].islower() for _, c in rows) / n
    avg = sum(len(c) for _, c in rows) / n
    print(f"{m:16}{n:8}{pt:16.0%}{lc:18.0%}{avg:11.0f}")

method            chunks  page-traceable  starts lowercase  avg chars
fixed                364              0%               66%        794
paragraph-aware      378              0%               34%        765
page-aware           511            100%               29%        564


In [8]:
# Cell B-signals + export — page-aware chunks with requirement signals
import json
MONTHS = r"(?:Jan|Feb|Mac|Mar|Apr|Mei|May|Jun|Jul|Ogos|Aug|Sep|Okt|Oct|Nov|Dis|Dec)[a-z]*\.?"
DATE_RE = re.compile(r"\b(?:\d{1,2}(?:st|nd|rd|th)?(?:\s*[-&]\s*\d{1,2}(?:st|nd|rd|th)?)?\s+)?" + MONTHS + r",?\s+\d{4}\b"
                     r"|\b" + MONTHS + r"\s+\d{1,2},\s+\d{4}\b", re.I)
DUR_RE = re.compile(r"(?:(?:at least|minimum of|maximum of|no later than|not later than|within|sekurang-kurangnya)\s+)?"
                    r"\d+(?:\s*(?:to|-)\s*\d+)?\s*(?:weeks?|months?|days?|years?|minggu|bulan|hari|tahun)\b"
                    r"(?:\s+(?:before|after|prior)\b)?", re.I)
TOPICS = {
 "deadline": r"deadline|must be submitted|tarikh akhir|no later than|before the",
 "duration": r"duration|period of|minimum of|tempoh|weeks? before|months? before",
 "credit":   r"credit transfer|credit exemption|pindah kredit|pengecualian kredit|\bcredits?\b|\bkredit\b",
 "eligibility": r"eligib|must be|required to|layak|syarat",
 "funding":  r"scholarship|stipend|funding|self-sponsored|biasiswa",
 "registration": r"course registration|register|pendaftaran",
 "calendar": r"revision|final examination|mid-semester|semester break|public holiday|cuti|ulangkaji|peperiksaan akhir",
 "visa_insurance": r"visa|insurance|insurans|passport",   # tagged for the RAI2 guardrail later
}
TOPIC_RE = {k: re.compile(v, re.I) for k, v in TOPICS.items()}

def extract_signals(chunk):
    norm = lambda m: re.sub(r"\s+", " ", m.group(0)).strip()
    all_dates = list(dict.fromkeys(norm(m) for m in DATE_RE.finditer(chunk)))
    durs = list(dict.fromkeys(norm(m) for m in DUR_RE.finditer(chunk)))[:4]
    years = [int(y) for d in all_dates for y in re.findall(r"\b(20\d{2})\b", d)]   # years from date expressions only
    return {"dates": all_dates[:6], "durations": durs,
            "topics": [k for k, r in TOPIC_RE.items() if r.search(chunk)],
            "max_year": max(years) if years else None}

def build_records(catalog, size=SIZE):
    recs = []
    for s in catalog:
        f = select_source(relative_path=s["rel"])
        sha = source_hash(f); is_pdf = f.suffix.lower() == ".pdf"
        for k, (page, chunk) in enumerate(page_aware_chunks(load_pages(f), size), 1):
            recs.append({"record_id": f'{s["id"]}-{k:04d}', "source_id": s["id"], "source": s["rel"],
                         "source_title": s["title"], "source_url": s["url"], "source_kind": s["kind"],
                         "page": page if is_pdf else None, "text": chunk,
                         "processing": f"page-aware-char-{size}", "source_sha256": sha,
                         "signals": extract_signals(chunk)})
    return recs

def build_profile(catalog, recs):
    prof = []
    for s in catalog:
        rs = [r for r in recs if r["source_id"] == s["id"]]
        years = [r["signals"]["max_year"] for r in rs if r["signals"]["max_year"]]
        my = max(years) if years else None
        fy = my if my else s["declared_year"]
        prof.append({"source_id": s["id"], "title": s["title"], "valid_for": s["valid_for"],
                     "chunks": len(rs), "empty_pages": INSPECTION[s["id"]]["empty_pages"],
                     "max_year_in_dates": my, "declared_year": s["declared_year"],
                     "freshness_basis": "dates in text" if my else "catalogue (no dates found)",
                     "possibly_outdated": fy < REFERENCE_YEAR,
                     "language": s["language"]})
    return prof

def write_outputs(out_dir, catalog):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    recs = build_records(catalog)
    assert recs and all(r["text"].strip() for r in recs), "Empty chunk produced"
    jl = out_dir / "prepared_text.jsonl"
    with jl.open("w", encoding="utf-8") as fh:
        for r in recs: fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    pf = out_dir / "source_profile.json"
    pf.write_text(json.dumps(build_profile(catalog, recs), indent=2, ensure_ascii=False), encoding="utf-8")
    return jl, pf, recs

prepared_path, profile_path, records = write_outputs(PROCESSED_DIR, SOURCES)
print("Chunks exported:", len(records), "| file:", prepared_path.name, prepared_path.stat().st_size, "bytes")
print("First record:", json.dumps({k: records[0][k] for k in ("record_id","source_title","page","processing")}))
print(json.dumps(json.loads(profile_path.read_text())[4], indent=1))

Chunks exported: 511 | file: prepared_text.jsonl 545778 bytes
First record: {"record_id": "S01-0001", "source_title": "FAI Academic Handbook 2025-2026", "page": 1, "processing": "page-aware-char-800"}
{
 "source_id": "S05",
 "title": "UTM Mobility Portal",
 "valid_for": "snapshot 2026-11-16; listings are 2022",
 "chunks": 8,
 "empty_pages": [],
 "max_year_in_dates": 2023,
 "declared_year": 2022,
 "freshness_basis": "dates in text",
 "possibly_outdated": true,
 "language": "en"
}


In [9]:
# B3 — basic output contract + content spot-check
assert prepared_path.exists() and prepared_path.stat().st_size > 0, "Prepared output missing/empty."
with prepared_path.open(encoding="utf-8") as fh:
    reloaded = [json.loads(l) for l in fh if l.strip()]
assert len(reloaded) == len(records)
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: PASS (exists, non-empty, reloadable, counts match)")

# content spot-check: a chunk that should contain a requirement
hit = next(r for r in reloaded if r["source_id"] == "S02" and "3" in " ".join(r["signals"]["durations"]))
print("\nSpot-check", hit["record_id"], "page", hit["page"], hit["signals"])
print(hit["text"][:300])

# requirement-spotting preview (supports later deadline/duration/availability filtering)
print("\nPer-source signal coverage:")
for s in SOURCES:
    rs = [r for r in reloaded if r["source_id"] == s["id"]]
    print(f'{s["id"]}: chunks={len(rs):3} with_dates={sum(bool(r["signals"]["dates"]) for r in rs):3} '
          f'with_durations={sum(bool(r["signals"]["durations"]) for r in rs):3} '
          f'deadline_topic={sum("deadline" in r["signals"]["topics"] for r in rs):3}')
print("\nSource freshness flags:")
for p in json.loads(profile_path.read_text()):
    print(p["source_id"], "max_year:", p["max_year_in_dates"], "declared:", p["declared_year"], "| possibly_outdated:", p["possibly_outdated"], "| empty_pages:", p["empty_pages"])

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 545778
Validation: PASS (exists, non-empty, reloadable, counts match)

Spot-check S02-0004 page 4 {'dates': [], 'durations': ['at least 3 months before'], 'topics': ['deadline', 'duration', 'eligibility', 'funding', 'visa_insurance'], 'max_year': None}
4
• Apply for study abroad
approval from DVC
(Academic &
International), UTM.
• Application MUST be
submitted at least 3
months before the
departure date
• UTM International will
assist student with the
application approval
including the budget
required (if any) upon
receiving a complete
form with s

Per-source signal coverage:
S01: chunks=245 with_dates=  2 with_durations=  1 deadline_topic=  0
S02: chunks= 13 with_dates=  0 with_durations=  3 deadline_topic=  3
S03: chunks= 11 with_dates= 11 with_durations= 11 deadline_topic=  0
S04: chunks=226 with_dates=  0 with_durations=  2 deadline_topic=  0
S05: chunks=  8 with_dates=  6 with_durations=  0 deadline_topic=  7


In [10]:
# B4.1 — register lineage for each source -> prepared output
for s in SOURCES:
    register_lineage(source=select_source(relative_path=s["rel"]), output=prepared_path,
                     processing=f"page-aware-char-{SIZE}; whitespace clean; signal tagging (heuristic)")

Registered: FAI_Academic_Handbook_2025_2026.pdf -> prepared_text.jsonl
Registered: UTM_International_Outbound_Pre-Departure_Briefing.pdf -> prepared_text.jsonl
Registered: UTM_Academic_Calendar_20262027.pdf -> prepared_text.jsonl
Registered: UTM_Academic_Rules_and_Regulations.pdf -> prepared_text.jsonl
Registered: utm_mobility_portal.txt -> prepared_text.jsonl
Registered: jasso_student_exchange_support.txt -> prepared_text.jsonl
Registered: shisu_china_link_2026.txt -> prepared_text.jsonl


In [11]:
# Cell 6A — common lineage manifest (source_id/version/rights taken from catalogue)
assert lineage_entries, "No lineage entries registered."
lineage_records = []
for entry in lineage_entries.values():
    source, output = entry["source"], entry["output"]
    cat = BY_REL[str(source.relative_to(RAW_DIR.resolve()))]
    lineage_records.append({
        "source_id": cat["id"], "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_title": cat["title"], "source_url": cat["url"],
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output), "processing_note": entry["processing"],
        "source_version": cat["valid_for"], "access_rights": cat["access"],
        "known_limitations": cat["note"], "status": "Prepared; quality review pending (Topic 4)"})
lineage_records.sort(key=lambda r: r["source_id"])
manifest = save_manifest(lineage_records)
print("Total lineage entries:", len(manifest["records"]))

Saved lineage: /content/week3_project/data/week3_manifest.json
Total lineage entries: 7


In [12]:
# Cell 6B — verify manifest
with (BASE_DIR / "data" / "week3_manifest.json").open(encoding="utf-8") as fh:
    saved = json.load(fh)["records"]
assert saved and len(saved) == len(lineage_entries)
for r in saved:
    assert source_hash(RAW_DIR / r["source_filename"]) == r["source_sha256"], r["source_filename"]
    assert source_hash(BASE_DIR / r["prepared_output"]) == r["prepared_sha256"]
print("COMMON LINEAGE VALIDATION: PASS | Verified records:", len(saved))

COMMON LINEAGE VALIDATION: PASS | Verified records: 7


## Part C — Executable V1 data pipeline
Same configuration as Part B (page-aware, 800 chars). Cell 7 = common runner (adapted so one run ingests *all* document sources); Cell 8A = documents pathway; Cell 9 = repeatability smoke test + check that Part C output equals Part B output.

In [13]:
# Cell 7 — common V1 pipeline utilities
from pathlib import Path
from datetime import datetime, timezone
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}
sha256_file = source_hash

def execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration):
    inputs = [Path(p) for p in sources]
    # INGEST
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    # INSPECT
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind; output_dir.mkdir(parents=True, exist_ok=True)
    # PREPARE / STORE
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    # VALIDATE
    if not outputs: raise ValueError("No prepared outputs were produced.")
    for o in outputs:
        if not o.is_file(): raise FileNotFoundError(f"Prepared output missing: {o}")
        if o.stat().st_size == 0: raise ValueError(f"Prepared output is empty: {o}")
    validation = {"output_count": len(outputs),
                  "output_sha256": [sha256_file(o) for o in outputs],
                  "output_bytes": [o.stat().st_size for o in outputs]}
    # RECORD LINEAGE
    record = {"pipeline_version": "week3-v1",
              "generated_at_utc": datetime.now(timezone.utc).isoformat(),
              "data_type": kind, "configuration": configuration, "processing_method": method,
              "sources": [{"path": str(p.relative_to(RAW_DIR)), "sha256": sha256_file(p)} for p in inputs],
              "outputs": [str(o.relative_to(BASE_DIR)) for o in outputs],
              "source_output_mapping": mappings, "inspection": inspection, "validation": validation,
              "source_version": "see source_version in data/week3_manifest.json",
              "access_rights": "Not verified - check before sharing"}
    (output_dir / "pipeline_manifest.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    return record

def register_and_run(kind, sources, inspect_fn, prepare_fn, configuration):
    runner = lambda: execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first
print("Cell 7 ready.")

Cell 7 ready.


In [14]:
# Cell 8A — documents pathway (PDF + TXT snapshots)
SOURCES_A = [select_source(relative_path=s["rel"]) for s in SOURCES]
CONFIG_A = {"method": "page-aware", "chunk_size": SIZE, "signals": "heuristic regex v1"}

def inspect_documents(inputs, cfg):
    out = {}
    for src in inputs:
        pages = load_pages(src)
        if not "".join(t for _, t in pages).strip():
            raise ValueError(f"No extractable text in {src.name}; scanned PDFs need OCR.")
        out[str(src.relative_to(RAW_DIR))] = {"pages": len(pages),
            "chars": sum(len(t) for _, t in pages),
            "empty_pages": [p for p, t in pages if len(t.strip()) < 30]}
    return out

def prepare_documents(inputs, out_dir, cfg):
    rels = {str(p.relative_to(RAW_DIR)) for p in inputs}
    catalog = [s for s in SOURCES if s["rel"] in rels]
    jl, pf, recs = write_outputs(out_dir, catalog)
    mappings = [{"source": s["rel"], "source_id": s["id"],
                 "chunks": sum(r["source_id"] == s["id"] for r in recs),
                 "outputs": [str(jl.relative_to(BASE_DIR)), str(pf.relative_to(BASE_DIR))]} for s in catalog]
    return [jl, pf], mappings, f"page-aware-char-{cfg['chunk_size']} + heuristic signal tagging"

run_a = register_and_run("documents", SOURCES_A, inspect_documents, prepare_documents, CONFIG_A)
print("Chunks per source:", {m["source_id"]: m["chunks"] for m in run_a["source_output_mapping"]})
print("Flagged empty pages:", {k: v["empty_pages"] for k, v in run_a["inspection"].items() if v["empty_pages"]})

SUCCESS: documents | Inputs: 7 | Outputs: 2
Chunks per source: {'S01': 245, 'S02': 13, 'S03': 11, 'S04': 226, 'S05': 8, 'S06': 3, 'S07': 5}
Flagged empty pages: {'pdf/FAI_Academic_Handbook_2025_2026.pdf': [53, 54, 55, 56]}


In [15]:
# Cell 9 — repeatability + provenance (+ Part B == Part C consistency)
assert PIPELINE_RUNS, "Run a Cell 8 pathway first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first, second = entry["first"], entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Mapping changed for {kind}"
    print(f"PASS: {kind} - source fingerprints, prepared outputs and mappings match")
b_hash = source_hash(PROCESSED_DIR / "prepared_text.jsonl")
c_hash = source_hash(PIPE_DIR / "documents" / "prepared_text.jsonl")
assert b_hash == c_hash, "Part C output differs from Part B output"
print("PASS: Part C prepared_text.jsonl is byte-identical to Part B")
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---


SUCCESS: documents | Inputs: 7 | Outputs: 2
PASS: documents - source fingerprints, prepared outputs and mappings match
PASS: Part C prepared_text.jsonl is byte-identical to Part B

All selected pipelines passed repeatability smoke checks.
